# Demo 7 - Impossible travel (distance and speed between sign-ins)

**Fast** (bounded to multi-country users) · **Pool:** Medium · **Visual:** speed bars

**The question:** did one account sign in from two places further apart than anybody could
physically travel in the time between?

For each user we take consecutive sign-ins, measure the real distance between the two
locations over the curve of the Earth, divide by the elapsed time, and flag anything moving
faster than an airliner.

This needs two things KQL is poor at: comparing each row against the previous row for the
same user, and calculating true distance between latitude and longitude pairs. Here both
are a few readable lines, and the chart comes free.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `LOOKBACK_DAYS` - how much sign-in history to examine.
- `IMPOSSIBLE_KMH` - the speed above which travel stops being plausible. 900 km/h is a
  little above an airliner's cruise, so anything faster means the same account was in use
  in two places at once.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 14
IMPOSSIBLE_KMH = 900   # faster than a commercial flight

## 3. Pull sign-ins that have coordinates, for users who crossed a border

`LocationDetails` arrives as a JSON string rather than a structured column, so we describe
its shape and unpack latitude, longitude and country out of it.

Then we bound the work. Checking every user's every hop would be slow and pointless, so we
first find the users who signed in from two or more countries and pull only their events.
Nobody else can possibly have travelled impossibly.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

geo = StructType([StructField("latitude", DoubleType()), StructField("longitude", DoubleType())])
loc = StructType([StructField("city", StringType()),
                  StructField("countryOrRegion", StringType()),
                  StructField("geoCoordinates", geo)])

df = data_provider.read_table("SigninLogs", WORKSPACE)
df = (df.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
        .filter(F.col("UserPrincipalName").isNotNull() &
                (F.trim(F.col("UserPrincipalName")) != ""))
        .withColumn("L", F.from_json("LocationDetails", loc))
        .withColumn("lat", F.col("L.geoCoordinates.latitude"))
        .withColumn("lon", F.col("L.geoCoordinates.longitude"))
        .withColumn("country", F.col("L.countryOrRegion"))
        .filter(F.col("lat").isNotNull() & F.col("lon").isNotNull()))

# Bound the work: only users seen in >=2 countries
multi = (df.groupBy("UserPrincipalName").agg(F.countDistinct("country").alias("c"))
           .filter(F.col("c") >= 2).select("UserPrincipalName"))
pdf = (df.join(multi, "UserPrincipalName")
         .select("UserPrincipalName","TimeGenerated","lat","lon","country","IPAddress")).toPandas()
print("candidate events:", len(pdf))

## 4. Work out the implied travel speed between consecutive sign-ins

For each user, sort their sign-ins by time and look at each consecutive pair: how far apart
were the two locations, and how long was the gap between them?

Distance is **geodesic**, meaning the true distance measured over the curve of the Earth
rather than a straight line through it. `geopy` does this properly. If the pool does not
have `geopy` installed, the cell falls back to a haversine calculation, which is within
about half a percent and more than good enough here.

Distance divided by elapsed time gives km/h, and anything above the threshold is flagged.

Two safeguards worth knowing about: a user's first sign-in has nothing to compare against
so it scores zero, and gaps shorter than a minute are treated as a full minute to stop the
division producing an absurd number.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

try:
    from geopy.distance import geodesic
    def km_between(a, b):
        return geodesic(a, b).km
except ImportError:
    # geopy is not guaranteed on every pool - haversine is within ~0.5% for this purpose.
    print("geopy unavailable on this pool - using a haversine fallback")
    def km_between(a, b):
        lat1, lon1, lat2, lon2 = map(np.radians, (a[0], a[1], b[0], b[1]))
        h = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
        return float(2 * 6371.0088 * np.arcsin(np.sqrt(h)))

if pdf.empty:
    # pandas .apply on an empty frame returns a DataFrame, not a Series, so guard first.
    print("No users signed in from 2+ countries in this window - raise LOOKBACK_DAYS.")
    flagged = pdf.assign(kmh=pd.Series(dtype="float64"))
else:
    pdf["TimeGenerated"] = pd.to_datetime(pdf["TimeGenerated"])
    pdf = pdf.sort_values(["UserPrincipalName","TimeGenerated"])
    g = pdf.groupby("UserPrincipalName")
    pdf["plat"], pdf["plon"] = g["lat"].shift(), g["lon"].shift()
    pdf["pt"] = g["TimeGenerated"].shift()

    def speed(r):
        if pd.isna(r["plat"]): return 0.0
        km = km_between((r["plat"], r["plon"]), (r["lat"], r["lon"]))
        hrs = max((r["TimeGenerated"] - r["pt"]).total_seconds()/3600, 1/60)
        return km / hrs

    pdf["kmh"] = pdf.apply(speed, axis=1)
    flagged = pdf[pdf["kmh"] > IMPOSSIBLE_KMH].sort_values("kmh", ascending=False)

print("impossible-travel hops:", len(flagged))
flagged[["UserPrincipalName","TimeGenerated","country","kmh"]].head(15)

## 5. Chart the worst offenders

One bar per user showing their fastest implied hop, with the threshold marked as a dashed
line.

**What to look for:** a bar at three or four times the threshold is usually a VPN or a
corporate proxy egressing in another country, not an attacker. The genuinely interesting
case is a modest overshoot from a country the user has no business being in. Check the raw
table above before you act on the chart.

In [ ]:
if not flagged.empty:
    top = (flagged.groupby("UserPrincipalName")["kmh"].max().nlargest(12))
    plt.figure(figsize=(11,5))
    plt.barh(top.index[::-1].astype(str), top.values[::-1], color="#c0392b")
    plt.axvline(IMPOSSIBLE_KMH, ls="--", color="black", label=f"{IMPOSSIBLE_KMH} km/h threshold")
    plt.xlabel("Max implied travel speed (km/h)"); plt.legend()
    plt.title("Users with impossible-travel sign-ins"); plt.tight_layout(); plt.show()
else:
    print("No impossible-travel detected in this window.")

## Why a notebook beats KQL here

This needs `lag()` across ordered rows AND a geodesic (Haversine) distance between lat/long pairs, then a speed calc. In KQL that's contortion; with `geopy` + pandas it's a few readable lines - and you get a chart for free.